In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../outputs', exist_ok=True)

def clean_financial_data(file_path):
    df = pd.read_excel(file_path)
    df = df.iloc[:, [0, 2, 3]].copy()
    df.columns = ['Item', 'Amount_2025', 'Amount_2024']
    df['Item'] = df['Item'].astype(str).str.strip().str.replace('：', '').str.replace(':', '').str.replace('\n', '')
    df = df.dropna(subset=['Item'])
    return df

In [3]:
df_p_balance = clean_financial_data('../data/parent/balance_sheet1.xlsx')

p_asset_25 = float(df_p_balance[df_p_balance['Item'].str.contains('资产总计')].iloc[0]['Amount_2025'])
p_liab_25 = float(df_p_balance[df_p_balance['Item'].str.contains('负债合计')].iloc[0]['Amount_2025'])

print(f"【母公司资产负债表】")
print(f"2025年母公司资产负债率：{(p_liab_25/p_asset_25)*100:.2f}%")

【母公司资产负债表】
2025年母公司资产负债率：19.41%


In [21]:
df_p_income = clean_financial_data('../data/parent/income_statement1.xlsx')

# 容错匹配：母公司可能叫“营业收入”或“营业总收入”
rev_row = df_p_income[df_p_income['Item'].str.contains('营业总收入')]
if rev_row.empty:
    rev_row = df_p_income[df_p_income['Item'].str.contains('营业收入')]

p_rev_25 = float(rev_row.iloc[0]['Amount_2025'])
p_profit_25 = float(df_p_income[df_p_income['Item'].str.contains('净利润')].iloc[0]['Amount_2025'])

print(f"【母公司利润表】")
print(f"2025年母公司营业收入：{p_rev_25/1e8:.2f} 亿元")
print(f"2025年母公司净利润：{p_profit_25/1e8:.2f} 亿元")
print(f"2025年母公司净利率：{(p_profit_25/p_rev_25)*100:.2f}%")

【母公司利润表】
2025年母公司营业收入：252.18 亿元
2025年母公司净利润：318.96 亿元
2025年母公司净利率：126.48%


In [22]:
df_p_cf = clean_financial_data('../data/parent/cash_flow1.xlsx')

p_ocf_25 = float(df_p_cf[df_p_cf['Item'].str.contains('经营活动产生的现金流量净额')].iloc[0]['Amount_2025'])
p_capex_25 = float(df_p_cf[df_p_cf['Item'].str.contains('购建固定资产')].iloc[0]['Amount_2025'])

print(f"【母公司现金流量表】")
print(f"2025年母公司经营现金流净额：{p_ocf_25/1e8:.2f} 亿元")
print(f"2025年母公司自由现金流：{(p_ocf_25-p_capex_25)/1e8:.2f} 亿元")
print(f"2025年母公司净现比：{p_ocf_25/p_profit_25:.2f}")

【母公司现金流量表】
2025年母公司经营现金流净额：162.45 亿元
2025年母公司自由现金流：111.99 亿元
2025年母公司净现比：0.51


In [23]:
# 引入合并报表的已知计算结果（单位：亿元）
cons_rev = 862.42
cons_profit = 349.49
cons_ocf = 605.63

# 打印最终对比
print(f"\n===== 集团合并 vs 母公司 2025 终极对比 =====")
print(f"【营收规模】合并 {cons_rev:.2f} 亿  vs  母公司 {p_rev_25/1e8:.2f} 亿")
print(f"【净利润额】合并 {cons_profit:.2f} 亿  vs  母公司 {p_profit_25/1e8:.2f} 亿")
print(f"【经营现金流】合并 {cons_ocf:.2f} 亿  vs  母公司 {p_ocf_25/1e8:.2f} 亿")

print("\n💡 核心洞察：")
print(f"母公司净利率高达 {(p_profit_25/p_rev_25)*100:.2f}%！")
print("原因：母公司自身发电收入虽然只有250多亿，但旗下子公司分红极高，导致其账面上拥有 235亿的【投资收益】！")


===== 集团合并 vs 母公司 2025 终极对比 =====
【营收规模】合并 862.42 亿  vs  母公司 252.18 亿
【净利润额】合并 349.49 亿  vs  母公司 318.96 亿
【经营现金流】合并 605.63 亿  vs  母公司 162.45 亿

💡 核心洞察：
母公司净利率高达 126.48%！
原因：母公司自身发电收入虽然只有250多亿，但旗下子公司分红极高，导致其账面上拥有 235亿的【投资收益】！
